# Notebook 2: Feature Preparation — Garbage In, Garbage Out

**Series:** Random Forests & Isolation Forests for Full-Stack Engineers  
**Prerequisites:** Notebook 1  
**Author:** [Farty Bobo](https://fartybobo.com)
**What you'll learn:** Why raw data is usually bad input for decision trees, and how to fix it

---

## The Software Engineering Analogy

You'd never pass raw, unvalidated user input directly to a database query. You sanitize it, normalize it, validate it first. The same principle applies in ML.

Decision trees have a specific weakness: they **split uniformly** along a feature's min-max range. If your data is heavily skewed (most values are clustered at one end), the tree wastes its splits exploring the sparse tail and misses the dense region where the action is.

It's like writing a binary search over a list that's 90% sorted in the first 10% of the range — you're spending index space on a sparse region.

---

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer, StandardScaler

pal = sns.color_palette('colorblind')
sns.set_palette(pal)

iris = load_iris()
print('Setup complete.')

## The Problem: Non-Uniform Distributions

Recall from Notebook 1: the decision tree algorithm tests candidate split thresholds **uniformly distributed** between the min and max of each feature.

Here's why that's a problem. Imagine a feature (e.g., request latency) that looks like this:

- 90% of values are between 0 and 50ms
- 10% of values are between 50ms and 5000ms (outliers)

If the tree picks 5 random thresholds uniformly between 0 and 5000:
- Thresholds might be: 1000, 2000, 3000, 4000, 500
- Only 1 of those 5 cuts falls in the region (0-500ms) where 90% of your data lives
- The algorithm effectively has **1 cut worth of resolution for 90% of your data**

Transforming to a uniform distribution fixes this: every threshold gets equal "coverage" of your data.

Let's make this concrete with a synthetic example:

In [ ]:
# Create synthetic data with a realistic, skewed distribution.
# Imagine this is "API response time" (ms) for two classes of requests:
# class 0 = normal requests, class 1 = slow/problematic requests

np.random.seed(42)
size = 20_000
rng = np.random.default_rng(seed=42)

# The raw feature follows an exponential distribution (common for latency/wait times)
# This mimics a system where most requests are fast but some have long tails
raw_feature = rng.exponential(size=size)  # values range from ~0 to ~15+, heavily skewed right

# The true separation is at the 50th percentile — a clean 50/50 split in quantile space
# We add a small logistic noise to simulate measurement noise
from sklearn.preprocessing import QuantileTransformer
q_tx = QuantileTransformer(output_distribution='uniform', random_state=42)
quantiles = q_tx.fit_transform(raw_feature.reshape(-1, 1)).flatten()

noise = rng.logistic(scale=0.1, size=size)   # small noise around the decision boundary
labels = ((quantiles + noise) > 0.5).astype(int)   # class boundary is at the median

print(f'Feature range: [{raw_feature.min():.2f}, {raw_feature.max():.2f}]')
print(f'Class 0 count: {(labels==0).sum():,}   Class 1 count: {(labels==1).sum():,}')

In [ ]:
# Visualize the problem: the same 5 random cuts in raw space vs quantile space

np.random.seed(7)  # fixed seed for reproducible cut positions

fig, axes = plt.subplots(1, 3, figsize=(15, 4), dpi=110)

# --- Panel 1: Raw distribution ---
bins = np.arange(0, 6, 0.05)
bin_width = bins[1] - bins[0]

h0, _ = np.histogram(raw_feature[labels==0], bins=bins)
h1, _ = np.histogram(raw_feature[labels==1], bins=bins)

axes[0].bar(bins[:-1], h0, width=bin_width, alpha=0.5, label='Class 0')
axes[0].bar(bins[:-1], h1, width=bin_width, alpha=0.5, label='Class 1')
axes[0].set_title('Raw Feature (Exponential Decay)', fontsize=11)
axes[0].set_xlabel('Feature Value')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Draw 5 random uniform cuts across the full range
feature_max = raw_feature.max()
random_cuts_raw = np.random.uniform(0, feature_max, size=5)
ylim = axes[0].get_ylim()[1]
for cut in random_cuts_raw:
    axes[0].axvline(cut, color='black', lw=1.5, alpha=0.8)
axes[0].text(0.65, 0.92, '5 uniform\nrandom cuts', transform=axes[0].transAxes,
             fontsize=9, ha='center', va='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

# --- Panel 2: Show where those same cuts land in quantile space ---
bins_q = np.linspace(0, 1, 101)
bin_width_q = bins_q[1] - bins_q[0]

qtiles0 = q_tx.transform(raw_feature[labels==0].reshape(-1, 1)).flatten()
qtiles1 = q_tx.transform(raw_feature[labels==1].reshape(-1, 1)).flatten()

hq0, _ = np.histogram(qtiles0, bins=bins_q)
hq1, _ = np.histogram(qtiles1, bins=bins_q)

axes[1].bar(bins_q[:-1], hq0, width=bin_width_q, alpha=0.5, label='Class 0')
axes[1].bar(bins_q[:-1], hq1, width=bin_width_q, alpha=0.5, label='Class 1')
axes[1].set_title('Same Cuts Projected to Quantile Space\n(Where cuts actually fall)', fontsize=11)
axes[1].set_xlabel('Percentile of Feature Value')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Project the raw cuts into quantile space
cuts_in_quantile_space = q_tx.transform(random_cuts_raw.reshape(-1, 1)).flatten()
ylim_q = axes[1].get_ylim()[1]
for cut_q in cuts_in_quantile_space:
    axes[1].axvline(cut_q, color='black', lw=1.5, alpha=0.8)
axes[1].text(0.65, 0.92, 'All 5 cuts\nclustered left!', transform=axes[1].transAxes,
             fontsize=9, ha='center', va='top',
             bbox=dict(boxstyle='round', facecolor='salmon', alpha=0.7))

# --- Panel 3: Uniform cuts on uniform distribution ---
axes[2].bar(bins_q[:-1], hq0, width=bin_width_q, alpha=0.5, label='Class 0')
axes[2].bar(bins_q[:-1], hq1, width=bin_width_q, alpha=0.5, label='Class 1')
axes[2].set_title('New Uniform Cuts on Transformed Data\n(Even coverage!)', fontsize=11)
axes[2].set_xlabel('Percentile of Feature Value')
axes[2].legend()
axes[2].grid(alpha=0.3)

# New cuts uniformly distributed in quantile space
uniform_cuts = np.random.uniform(0, 1, size=5)
for cut in uniform_cuts:
    axes[2].axvline(cut, color='black', lw=1.5, alpha=0.8)
axes[2].text(0.65, 0.92, '5 uniform\ncuts on\nuniform data', transform=axes[2].transAxes,
             fontsize=9, ha='center', va='top',
             bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))

plt.suptitle('Why Data Distribution Matters for Decision Trees', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### What You're Seeing

- **Panel 1 (raw):** The data is exponentially distributed. 5 random cuts span the full range.
- **Panel 2 (same cuts, quantile space):** All 5 cuts pile up in the leftmost 10% of the data! The other 90% of your data has zero cut resolution.
- **Panel 3 (transformed):** After quantile transformation, 5 cuts spread evenly across all 100 percentiles. Each cut covers ~20% of your data.

Same 5 cuts — radically different coverage. **Transforming to uniform distribution is free accuracy.**

---

## The Fix: Quantile Transformation

The **QuantileTransformer** maps your data to a uniform (or normal) distribution by rank:
- The smallest value → 0.0
- The median value → 0.5
- The largest value → 1.0
- Everything in between → its percentile rank

It's exactly like computing `percentile_rank(value)` in SQL:
```sql
PERCENT_RANK() OVER (ORDER BY response_time)
```

The tradeoff: you lose the absolute values and intervals between values. A latency of 100ms vs 200ms vs 10000ms all become their percentile ranks. For classification this is usually fine — you care about *relative* ordering, not absolute magnitude.

In [ ]:
# Concrete example: fit a QuantileTransformer and see what it does to values

# Create a small, easy-to-understand example
sample_latencies = np.array([1, 2, 3, 5, 10, 20, 50, 100, 500, 2000]).reshape(-1, 1)

qt = QuantileTransformer(output_distribution='uniform', random_state=42)
qt.fit(sample_latencies)  # learns the distribution from training data

transformed = qt.transform(sample_latencies)

print('Original → Transformed (percentile rank):')
for orig, trans in zip(sample_latencies.flatten(), transformed.flatten()):
    bar = '█' * int(trans * 30)
    print(f'  {orig:5.0f}ms  →  {trans:.2f}  {bar}')

In [ ]:
# Now let's MEASURE the impact: train two trees (raw vs transformed) and compare
# We'll use the synthetic exponential data from earlier.

# Split into train and test sets (80/20 split)
# This is critical: never test on data you trained on! (Notebook 1's overfitting warning.)
X_raw = raw_feature.reshape(-1, 1)  # 2D array: (n_samples, 1_feature)
y     = labels

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42
)

# Fit transformer ONLY on training data, then apply to test data.
# NEVER fit on test data — that's "data leakage" (like reading answers before the exam).
qt_train = QuantileTransformer(output_distribution='uniform', random_state=42)
X_train_transformed = qt_train.fit_transform(X_train_raw)   # fit+transform on train
X_test_transformed  = qt_train.transform(X_test_raw)         # transform-only on test

# Train on raw data
tree_raw = DecisionTreeClassifier(max_depth=5, random_state=42)
tree_raw.fit(X_train_raw, y_train)

# Train on transformed data
tree_transformed = DecisionTreeClassifier(max_depth=5, random_state=42)
tree_transformed.fit(X_train_transformed, y_train)

# Evaluate on test set
score_raw         = tree_raw.score(X_test_raw, y_test)
score_transformed = tree_transformed.score(X_test_transformed, y_test)

print(f'Test accuracy — Raw data:         {score_raw:.1%}')
print(f'Test accuracy — Transformed data: {score_transformed:.1%}')
print(f'Improvement:                      +{score_transformed - score_raw:.1%}')

---

## Train/Test Split: The Key Concept

Before we go further, let's make the train/test split concept crystal clear.

```
All your labeled data
         │
    ┌────┴─────┐
    │          │
 80% Train  20% Test
    │          │
 fit(X,y)   score(X,y)  ← model NEVER saw these during training
```

The test set is your "never-before-seen data" simulation. If your model does well on the test set, you have evidence it generalizes.

> **Critical rule:** NEVER use test data during training, feature selection, or any preprocessing that involves fitting a transformer. Fitting `QuantileTransformer` on test data would be "leaking" information about the test set into your model — it's cheating, and it gives you falsely optimistic accuracy numbers.

In [ ]:
# Let's visualize the train/test split to make it concrete

iris_df = pd.DataFrame(iris.data, columns=iris.feature_names)
iris_df['species'] = [iris.target_names[t] for t in iris.target]

X_train, X_test, y_train_iris, y_test_iris = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target
    # stratify=iris.target ensures proportional class representation in both splits
)

print(f'Train set: {X_train.shape[0]} samples ({X_train.shape[0]/len(iris.data):.0%})')
print(f'Test set:  {X_test.shape[0]} samples ({X_test.shape[0]/len(iris.data):.0%})')
print()
print('Class distribution in train:')
for cls, name in enumerate(iris.target_names):
    n = (y_train_iris == cls).sum()
    print(f'  {name}: {n}')
print('Class distribution in test:')
for cls, name in enumerate(iris.target_names):
    n = (y_test_iris == cls).sum()
    print(f'  {name}: {n}')

## Other Preprocessing Considerations

### StandardScaler (Z-score normalization)

An alternative to QuantileTransformer. It transforms each feature to have mean=0 and std=1:

```
x_scaled = (x - mean) / std
```

This is like normalizing a score by how many standard deviations it is from the average. Useful for algorithms that care about distance between points (not decision trees specifically, but good to know).

Decision trees are actually **scale-invariant** — multiplying all values by 1000 doesn't change which split threshold is best. But QuantileTransformer helps because it changes the *distribution shape*, not just the scale.

### Categorical Variables

Decision trees split on numeric thresholds. Categorical variables (`['red', 'green', 'blue']`) must be encoded as numbers. The naive approach (1, 2, 3) is problematic:

```
Problem: encoding 'red'=1, 'green'=2, 'blue'=3
         implies red < green < blue, which is meaningless.
         To isolate 'green', the tree needs TWO cuts: x>1 AND x<3.
         This is expensive and may not work with limited depth.
```

Better: **one-hot encoding** — create a separate binary feature for each category:

| is_red | is_green | is_blue |
|---|---|---|
| 1 | 0 | 0 |
| 0 | 1 | 0 |
| 0 | 0 | 1 |

Now each category is its own binary feature — one cut separates it perfectly.

### Handling Missing Values (NaN/NULL)

Sklearn's decision tree does NOT handle NaN by default. Options:
1. **Drop rows** — only if missing data is rare and random
2. **Impute with median/mean** — if missing = "not recorded"
3. **Impute with out-of-range sentinel** — if missing IS meaningful (covered in Notebook 4)

---

## Putting It Together: A Full Preprocessing Pipeline

In [ ]:
# sklearn has a Pipeline object that chains preprocessing + model together.
# This is like middleware in Express.js — each step transforms the data before passing it on.
# The big win: Pipeline correctly handles train/test data separation automatically.

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import QuantileTransformer
from sklearn.tree import DecisionTreeClassifier

# Build the pipeline: step 1 = transform features, step 2 = classify
pipeline = Pipeline([
    ('scaler', QuantileTransformer(output_distribution='uniform', random_state=42)),
    ('classifier', DecisionTreeClassifier(max_depth=4, random_state=42)),
])

# The pipeline's fit() correctly:
# 1. Fits the transformer on X_train only
# 2. Transforms X_train
# 3. Fits the classifier on transformed X_train
pipeline.fit(X_train, y_train_iris)

# The pipeline's score() correctly:
# 1. Transforms X_test using the already-fitted transformer (no re-fitting!)
# 2. Predicts on transformed X_test
# 3. Returns accuracy
train_acc = pipeline.score(X_train, y_train_iris)
test_acc  = pipeline.score(X_test, y_test_iris)

print(f'Train accuracy: {train_acc:.1%}')
print(f'Test accuracy:  {test_acc:.1%}')
print()
print('The gap between train and test accuracy is a sign of overfitting.')
print('A small gap means good generalization.')

---

## Summary

| Concept | What it is | Why it matters |
|---|---|---|
| **Feature distribution** | The shape of how values are spread | Skewed data wastes split resolution |
| **QuantileTransformer** | Maps data to uniform distribution by rank | Decision trees get even coverage across all data |
| **Train/test split** | Evaluate on data the model never saw | Measures true generalization, not memorization |
| **Data leakage** | Fitting preprocessors on test data | Gives falsely optimistic accuracy |
| **Pipeline** | Chains preprocessing + model | Handles train/test separation correctly |

---

## What's Next

**Notebook 3: Random Forests** — A single decision tree has high variance (small data changes → big tree changes). Random Forests fix this by combining hundreds of trees. We'll see why this works and how to tune it.